In [ ]:
# Task 1: Import Packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow.keras.models as models
import tensorflow.keras.layers as layers
import tensorflow.keras.optimizers as optimizers
# Task 2: Load the Data
df = pd.read_csv("/usercode/btc.csv", index_col='Date',  parse_dates=True)
df.head(5)
# Task 3: Explore the Data

# Display the summary statistics
df.describe()
# Display the line plot
plt.plot(df.index, df['Open'])
plt.xlabel('Date')
plt.ylabel('Open Price')
plt.title('Bitcoin Open Price Over Time')
plt.show()
# Task 4: Manipulate the Data

df1 = df['Open']
scaler = MinMaxScaler()
df2 = scaler.fit_transform(df1.values.reshape(-1, 1))
df2 = pd.DataFrame(df2)
df2 = df2[0]
# Task 5: Generate the Input-Output Pairs

# Define a function to convert DataFrame to X and y pairs with a specified window size.
def df_to_X_y(df, window_size=15):
    # Convert DataFrame to a NumPy array for easier manipulation.
    df_as_np = df.to_numpy()
    
    # Initialize empty lists to store input sequences and corresponding labels.
    X = []
    y = []
    
    # Iterate over the array to create input-output pairs.
    for i in range(len(df_as_np) - window_size):
        # Create a sequence of length 'window_size' for input.
        row = [[a] for a in df_as_np[i:i+window_size]]
        
        # Append the input sequence to the list.
        X.append(row)
        
        # Extract the label corresponding to the input sequence.
        label = df_as_np[i+window_size]
        
        # Append the label to the list.
        y.append(label)
    
    # Convert lists to NumPy arrays and return them as X and y pairs.
    return np.array(X), np.array(y)

# Define a constant for the window size.
WINDOW_SIZE = 15

# Call the function with the DataFrame and window size to get X1, y1 pairs.
X1, y1 = df_to_X_y(df2, WINDOW_SIZE)
# Task 6: Create and Compile the LSTM Model

# Create a sequential model for the univariate LSTM.
model1 = models.Sequential()

# Add an input layer with the specified shape (WINDOW_SIZE, 1).
model1.add(layers.InputLayer((WINDOW_SIZE, 1)))

# Add an LSTM layer with 64 units.
model1.add(layers.LSTM(64))

# Add a dense layer with 8 units and ReLU activation function.
model1.add(layers.Dense(8, activation='relu'))

# Add a dense layer with 1 unit and linear activation function (for regression).
model1.add(layers.Dense(1, activation='linear'))

# Compile the model using the mean squared error loss and the Adam optimizer with a learning rate of 0.0001.
model1.compile(loss='mean_squared_error', optimizer=optimizers.Adam(learning_rate=0.0001))
# Task 7: Train the LSTM Model

split_index = int(len(df2) * 0.8)
X_train, y_train = X1[:split_index], y1[:split_index]
X_val, y_val = X1[split_index:], y1[split_index:]
model1.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val))
# Task 8: Predict the Training Data

train_predictions = model1.predict(X1).flatten()
train_dates = df.index[WINDOW_SIZE:len(df2)]
train_results = pd.DataFrame(data={'Date': train_dates, 'Train Predictions': train_predictions, 'Actuals': y1.flatten()})
train_results
# Task 9: Generate Future Dates for Predictions

num_future_steps = 30
last_date = pd.to_datetime(df.index[-1])
future_dates = pd.date_range(last_date + pd.DateOffset(days=1), periods=num_future_steps)
future_dates
# Task 10: Make Predictions for Future Dates

future_predictions = []
recent_data = X1[-1].reshape(1, WINDOW_SIZE, 1)
for _ in range(num_future_steps):
    prediction = model1.predict(recent_data)[0][0]
    future_predictions.append(prediction)
    prediction = np.array([prediction])
    recent_data = np.append(recent_data[:, 1:, :], [[prediction]], axis=1)

future_results = pd.DataFrame(data={'Date': future_dates, 'Predictions': future_predictions})
future_results
# Task 11: Inverse Transform Predictions

# Function to inverse transform DataFrame
def inverse_transform_dataframe(df, scaler):
    df_inverse = df.copy()
    for col in df.columns:
        if col != 'Date':
            df_inverse[col] = scaler.inverse_transform(df[col].values.reshape(-1, 1)).flatten()
    return df_inverse

# Inverse transform predictions
train_results = inverse_transform_dataframe(train_results, scaler)
future_results = inverse_transform_dataframe(future_results, scaler)

# Combine train_results with future_results
combined_results = pd.concat([train_results, future_results], axis=0)

combined_results.to_csv("/usercode/combined_results.csv")
future_results.to_csv("/usercode/future_results.csv")
